In [2]:
import pandas as pd

columns = [
    "user_id",
    "date_time",
    "latitude",
    "longitude",
    "species",
    "fish_count",
    "avg_size",
    "temperature",
    "weather",
    "wind_speed",
    "time_of_day",
    "season",
    "success"
]

df = pd.DataFrame(columns=columns)

### What Each Column Means (IMPORTANT for your understanding)
- species: Red Drum, Flounder, Weakfish (gray trout), Spotted Seatrout (speckled trout), and Striped Bass
- fish_count: how many caught
- success: 1 if caught fish, 0 if none (VERY important feature)
- time_of_day: morning / afternoon / evening
- season: spring / summer / fall / winter

In [3]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random

def generate_data(n=1000):
    data = []

    species_list = ["striped bass", "weakfish", "spotted seatrout", "flounder", "red_drum"]
    weather_list = ["sunny", "cloudy", "rainy"]
    time_list = ["morning", "afternoon", "evening"]
    season_list = ["spring", "summer", "fall", "winter"]

    for i in range(n):
        temp = np.random.uniform(40, 90)
        wind = np.random.uniform(0, 20)
        time_of_day = random.choice(time_list)
        weather = random.choice(weather_list)
        season = random.choice(season_list)
        species = random.choice(species_list)

        # 🎯 DEFINE "GOOD CONDITIONS"
        success_prob = 0.3

        if 60 <= temp <= 75:
            success_prob += 0.2
        if wind < 10:
            success_prob += 0.2
        if time_of_day == "morning":
            success_prob += 0.2
        if weather == "cloudy":
            success_prob += 0.1

        success = np.random.rand() < success_prob

        fish_count = np.random.randint(1, 5) if success else 0
        avg_size = np.random.uniform(10, 30) if success else 0

        row = [
            random.randint(1, 50),
            datetime.now() - timedelta(days=np.random.randint(0, 365)),
            np.random.uniform(33, 37),
            np.random.uniform(-80, -75),
            species,
            fish_count,
            avg_size,
            temp,
            weather,
            wind,
            time_of_day,
            season,
            int(success)
        ]

        data.append(row)

    return pd.DataFrame(data, columns=columns)

df = generate_data(2000)
df.head()

,user_id,date_time,latitude,longitude,species,fish_count,avg_size,temperature,weather,wind_speed,time_of_day,season,success
0,27,2025-04-12 12:03:21.480117,36.175788,-76.579029,red_drum,0,0.000000,49.298650,sunny,16.813554,evening,summer,0
1,30,2025-10-30 12:03:21.480396,34.645127,-79.913911,striped bass,0,0.000000,84.293373,rainy,0.347045,afternoon,summer,0
2,34,2025-12-07 12:03:21.480514,33.434286,-75.250518,weakfish,4,29.180259,60.532970,cloudy,13.706655,morning,summer,1
3,32,2025-12-19 12:03:21.480573,34.879231,-75.858986,weakfish,0,0.000000,50.621943,cloudy,18.553599,afternoon,winter,0
4,8,2025-12-17 12:03:21.480651,33.789458,-76.816233,red_drum,1,17.110127,69.754840,sunny,3.903467,evening,fall,1


In [4]:
df["success"].value_counts()

success
1    1113
0     887
Name: count, dtype: int64

In [5]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["success", "date_time"])
y = df["success"]

# Convert categorical variables
X = pd.get_dummies(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print("Accuracy:", accuracy)

Accuracy: 1.0


In [7]:
import pandas as pd

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.coef_[0]
})

importance = importance.sort_values(by="importance", ascending=False)
print(importance.head(10))

                feature  importance
4              avg_size    1.358155
3            fish_count    0.250878
2             longitude    0.090316
17  time_of_day_morning    0.016262
11     species_weakfish    0.008121
19        season_spring    0.006169
12       weather_cloudy    0.005606
13        weather_rainy    0.002879
5           temperature    0.002661
21        season_winter    0.001851


In [9]:
print(df.groupby("time_of_day")["success"].mean())

time_of_day
afternoon    0.481536
evening      0.522828
morning      0.670807
Name: success, dtype: float64


In [11]:
print(df.groupby("weather")["success"].mean())

weather
cloudy    0.625879
rainy     0.545741
sunny     0.491603
Name: success, dtype: float64
